### Lakebase Project (shared Autoscaling)

Provisions the **single, shared Lakebase Autoscaling project** (`w.postgres`) that every
Lakebase-consuming stage and app in this bundle now uses.

Project layout:

- **Project**: `${CATALOG}-caspers`  (one per CATALOG)
- **Branch**: `production` (default)
- **Endpoint**: `primary` (default; created with the project)
- **Databases inside the project** (each created by its owning downstream stage):
  - `caspers_refund`    — created by `stages/lakebase.ipynb`
  - `caspers_complaint` — created by `stages/complaint_lakebase.ipynb`
  - `caspers_ops`       — created by `stages/operational_lakebase.ipynb`
  - (`databricks_postgres` ships with the project; we don't use it for app data)

The project creator is automatically granted superuser in every database in this project,
so DDL from any downstream stage (running as the bundle deployer) just works.

**This stage only creates the project and waits for the endpoint to be ready.**  Database
creation, schema DDL, synced-table wiring, and app deploys all live in their per-component
stages.  Every downstream Lakebase stage declares `Lakebase_Project` in its `depends_on`.

In [ ]:
%pip install --upgrade "databricks-sdk>=0.81.0"

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re

CATALOG = dbutils.widgets.get("CATALOG")

# Project id sanitised to the API's [a-z0-9-] alphabet.  All four databases
# live inside this single project; isolation comes from per-database grants,
# not per-project ones.
PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-caspers".lower())
PROJECT_RESOURCE_NAME = f"projects/{PROJECT_ID}"
BRANCH_PATH = f"{PROJECT_RESOURCE_NAME}/branches/production"
ENDPOINT_PATH = f"{BRANCH_PATH}/endpoints/primary"

print(f"CATALOG          = {CATALOG}")
print(f"PROJECT_ID       = {PROJECT_ID}")
print(f"BRANCH_PATH      = {BRANCH_PATH}")
print(f"ENDPOINT_PATH    = {ENDPOINT_PATH}")

##### Create or reuse the shared Autoscale project

In [ ]:
import sys, time
sys.path.append('../utils')
from uc_state import add
import status

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import Project, ProjectSpec

w = WorkspaceClient()

is_new = False
try:
    project = w.postgres.get_project(name=PROJECT_RESOURCE_NAME)
    status.reuse(f"Reusing existing Lakebase project: {PROJECT_ID}")
except Exception:
    is_new = True
    status.info(f"Creating Lakebase Autoscale project: {PROJECT_ID}")
    # create_project returns a CreateProjectOperation (Google LRO-style); it
    # has no .result(), and the project's resource name is deterministically
    # `projects/{PROJECT_ID}`.  The endpoint readiness loop below waits for
    # provisioning to finish.
    w.postgres.create_project(
        project=Project(
            spec=ProjectSpec(
                display_name=f"Caspers Kitchens ({CATALOG})",
                pg_version="17",
            )
        ),
        project_id=PROJECT_ID,
    )
    add(CATALOG, "postgres_projects", {
        "project_id": PROJECT_ID,
        "name": PROJECT_RESOURCE_NAME,
    })
    status.ok(f"Created project: {PROJECT_RESOURCE_NAME}")

##### Wait for the primary endpoint to be RUNNING

For freshly created projects this typically takes 1–2 min.  For reused
projects the endpoint is already up, so we still GET it (one call) to
surface the host in the stage logs and validate the path.

In [ ]:
if is_new:
    status.info("Waiting for endpoint to be ready...")
    for attempt in range(40):
        try:
            ep = w.postgres.get_endpoint(name=ENDPOINT_PATH)
            state = str(ep.status.current_state) if ep.status else "UNKNOWN"
            if any(s in state for s in ("RUNNING", "AVAILABLE", "ACTIVE")):
                status.ok(f"Endpoint ready (state={state})")
                break
            print(f"  [{attempt+1}/40] state={state}, retrying in 15s...")
        except Exception as e:
            print(f"  [{attempt+1}/40] not yet reachable: {e}")
        time.sleep(15)
    else:
        raise TimeoutError(
            f"Endpoint {ENDPOINT_PATH} did not start within 10 minutes"
        )
else:
    ep = w.postgres.get_endpoint(name=ENDPOINT_PATH)
    status.ok(f"Endpoint ready (state={ep.status.current_state})")

host = ep.status.hosts.host
status.ok(f"Lakebase project stage complete")
print(f"   Project:  {PROJECT_RESOURCE_NAME}")
print(f"   Endpoint: {ENDPOINT_PATH}")
print(f"   Host:     {host}")